# Session 3 — Track A: Production RAG Capstone — McKinsey Knowledge Assistant (Instructor)

In this capstone, students build a **production-grade RAG service** that serves as a **McKinsey Knowledge Assistant** -- a system that answers consulting questions by retrieving from a curated knowledge base of industry reports, strategic frameworks, and engagement playbooks.

This integrates everything from Days 1-3: embeddings, vector stores, chunking, query transformation, caching, monitoring, and evaluation -- applied to a realistic McKinsey consulting use case.

> **Instructor Note:** This notebook contains fully solved versions of all 4 milestones with detailed approach explanations. Use this as a reference during the lab and for students who get stuck.

## Capstone Objectives

By the end of this capstone, students will have built a **McKinsey Knowledge Assistant** comprising:

1. A **document ingestion pipeline** that indexes McKinsey consulting content (frameworks, playbooks, industry reports) with smart chunking and rich metadata (practice area, industry, engagement type, confidentiality level)
2. A **retrieval engine** with query expansion and reranking, tuned for consulting questions from partners and engagement managers
3. A **generation layer** that produces executive-ready answers with citations, MECE structure, and strategic relevance
4. A **production wrapper** with caching, monitoring, and evaluation measuring consulting quality (actionability, strategic depth, executive readiness)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
import json
import time
import hashlib
import numpy as np
from datetime import datetime
from collections import defaultdict
from typing import List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

---
## Milestone 1: Document Ingestion Pipeline

Build a pipeline that ingests McKinsey consulting knowledge assets -- frameworks, playbooks, industry reports, and CEO briefing templates -- applies smart chunking, embeds, and indexes them in ChromaDB.

Each document carries rich consulting metadata: practice area (e.g., Operations, M&A, Digital), industry vertical, engagement type, and confidentiality level. This metadata enables filtered retrieval by context.

> **Approach:** We create a class that wraps the full ingestion flow. Documents are chunked with `RecursiveCharacterTextSplitter`, and each chunk inherits metadata from its parent doc (source, practice area, industry) plus gets new metadata (chunk_index, total_chunks). All chunks are batch-embedded and stored in ChromaDB.

In [ ]:
# Milestone 1 — SOLUTION: Document Ingestion Pipeline

class DocumentIngester:
    def __init__(self, collection_name="mckinsey_knowledge", chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50"))):
        self.embed_client = openai.OpenAI()
        self.chroma = chromadb.Client()
        self.collection = self.chroma.create_collection(
            name=collection_name, metadata={"hnsw:space": "cosine"}
        )
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size, chunk_overlap=chunk_overlap,
            separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
        )

    def ingest(self, documents: List[Dict]) -> Dict:
        """Ingest documents. Each doc: {text, source, metadata}."""
        all_chunks = []
        all_metadatas = []
        all_ids = []
        
        for doc_idx, doc in enumerate(documents):
            chunks = self.splitter.split_text(doc["text"])
            for chunk_idx, chunk_text in enumerate(chunks):
                chunk_id = f"doc{doc_idx}_chunk{chunk_idx}"
                meta = {
                    "source": doc.get("source", "unknown"),
                    "chunk_index": chunk_idx,
                    "total_chunks": len(chunks),
                    **doc.get("metadata", {})
                }
                all_chunks.append(chunk_text)
                all_metadatas.append(meta)
                all_ids.append(chunk_id)
        
        # Batch embed
        embeddings = [
            e.embedding for e in self.embed_client.embeddings.create(
                model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=all_chunks
            ).data
        ]
        
        # Store in ChromaDB
        self.collection.add(
            documents=all_chunks, embeddings=embeddings,
            ids=all_ids, metadatas=all_metadatas
        )
        
        stats = {
            "documents_ingested": len(documents),
            "total_chunks": len(all_chunks),
            "avg_chunk_size": round(np.mean([len(c) for c in all_chunks]), 1),
            "collection_size": self.collection.count()
        }
        print(f"Ingested {stats['documents_ingested']} docs -> {stats['total_chunks']} chunks")
        return stats


# McKinsey consulting knowledge base
test_docs = [
    {
        "text": "McKinsey's post-merger integration (PMI) framework emphasizes three pillars: capturing synergies within the first 100 days, aligning organizational culture through structured leadership workshops, and establishing a clean-room integration management office (IMO). Research across 200+ acquisitions shows that companies capturing 80% of synergies in year one outperform peers by 15% in total shareholder returns. The key accelerator is appointing a dedicated integration leader at the C-suite level.",
        "source": "pmi_framework_2024.md",
        "metadata": {"practice_area": "M&A", "industry": "Cross-industry", "engagement_type": "Framework", "confidentiality": "Internal"}
    },
    {
        "text": "Digital transformation in manufacturing requires a phased approach: Phase 1 -- Assess digital maturity across the value chain using McKinsey's Digital Quotient (DQ) diagnostic. Phase 2 -- Prioritize use cases by impact and feasibility (typically predictive maintenance, demand sensing, and quality analytics yield highest ROI). Phase 3 -- Build a scalable data platform and upskill the workforce. Clients who follow this sequencing achieve 2-3x faster time-to-value versus those who pursue ad hoc digitization.",
        "source": "digital_manufacturing_playbook.md",
        "metadata": {"practice_area": "Digital", "industry": "Manufacturing", "engagement_type": "Playbook", "confidentiality": "Client-ready"}
    },
    {
        "text": "Operations excellence programs typically deliver 15-25% cost reduction in the first 18 months. McKinsey's approach begins with a diagnostic spanning procurement, production, and logistics. Lean management principles are combined with advanced analytics to identify waste. A critical success factor is embedding a performance management infrastructure -- daily management systems, KPI cascades, and capability-building academies -- to sustain gains beyond the engagement.",
        "source": "ops_excellence_guide.md",
        "metadata": {"practice_area": "Operations", "industry": "Cross-industry", "engagement_type": "Implementation Guide", "confidentiality": "Internal"}
    },
    {
        "text": "CEO briefing templates should follow the Situation-Complication-Resolution (SCR) structure. Open with a one-sentence situation framing the strategic context. Introduce the complication -- the specific challenge or opportunity requiring action. Close with the resolution -- McKinsey's recommended path forward with 2-3 actionable next steps. Appendices should include supporting data exhibits, risk matrices, and implementation timelines. Keep the core narrative to 5-7 pages maximum.",
        "source": "ceo_briefing_template.md",
        "metadata": {"practice_area": "Leadership", "industry": "Cross-industry", "engagement_type": "Template", "confidentiality": "Internal"}
    },
    {
        "text": "Supply chain resilience has become a board-level priority following recent global disruptions. McKinsey's supply chain diagnostic evaluates five dimensions: supplier diversification, inventory buffering strategy, logistics network flexibility, digital visibility (control tower maturity), and workforce agility. Best-in-class companies maintain dual sourcing for critical components, hold 4-6 weeks of safety stock for high-risk SKUs, and use real-time demand-supply matching algorithms to rebalance dynamically.",
        "source": "supply_chain_resilience_report.md",
        "metadata": {"practice_area": "Operations", "industry": "Manufacturing", "engagement_type": "Industry Report", "confidentiality": "Client-ready"}
    }
]

ingester = DocumentIngester()
stats = ingester.ingest(test_docs)
print(stats)

---
## Milestone 2: Advanced Retrieval Engine

Build a retrieval engine that handles the types of questions McKinsey consultants and partners ask: strategic queries that may span multiple practice areas, industry-specific questions, and requests for specific frameworks or best practices.

The engine must support query expansion (since consultants phrase the same question in many ways), deduplication, and LLM reranking to surface the most strategically relevant content.

> **Approach:** We expand each query into 3 variants using the LLM, retrieve candidates for each from ChromaDB, deduplicate by document text, then use the LLM to score relevance 0-10. The final results are sorted by score.

In [ ]:
# Milestone 2 — SOLUTION: Advanced Retrieval Engine

class RetrievalEngine:
    def __init__(self, collection):
        self.collection = collection
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.embed_client = openai.OpenAI()

    def _expand_query(self, query, n=3):
        response = self.llm.invoke([
            SystemMessage(content=f"Generate {n} alternative versions of this consulting question for search. Return as a JSON list of strings. Focus on different phrasings a McKinsey consultant might use."),
            HumanMessage(content=query)
        ])
        try:
            alts = json.loads(response.content)
        except:
            alts = []
        return [query] + alts

    def _retrieve_raw(self, queries, per_query_k=4):
        seen = set()
        candidates = []
        for q in queries:
            q_emb = self.embed_client.embeddings.create(
                model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[q]
            ).data[0].embedding
            results = self.collection.query(query_embeddings=[q_emb], n_results=per_query_k)
            for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
                if doc not in seen:
                    seen.add(doc)
                    candidates.append({"text": doc, "metadata": meta})
        return candidates

    def _rerank(self, query, candidates):
        scored = []
        for c in candidates:
            response = self.llm.invoke([
                SystemMessage(content="Rate relevance 0-10 for a McKinsey consulting knowledge assistant. Consider strategic relevance, specificity, and actionability. Return ONLY the number."),
                HumanMessage(content=f"Consulting question: {query}\nKnowledge base excerpt: {c['text']}")
            ])
            try:
                score = int(response.content.strip())
            except:
                score = 5
            scored.append({**c, "score": score})
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored

    def retrieve(self, query, k=5) -> List[Dict]:
        queries = self._expand_query(query)
        candidates = self._retrieve_raw(queries)
        ranked = self._rerank(query, candidates)
        return ranked[:k]


# Test with a McKinsey consulting question
engine = RetrievalEngine(ingester.collection)
results = engine.retrieve("What is McKinsey's approach to post-merger integration?")
print("Retrieved knowledge base excerpts:")
for r in results:
    print(f"  {r['score']}/10 [{r['metadata']['source']}] ({r['metadata'].get('practice_area', 'N/A')}) {r['text'][:80]}...")

---
## Milestone 3: Generation Layer with Citations

Build the generation component that produces executive-ready consulting answers. Responses must cite sources (framework documents, playbooks, reports), follow a structured format suitable for partner review, and flag when the knowledge base lacks sufficient information to answer confidently.

> **Approach:** We format retrieved chunks with source tags, instruct the LLM to cite sources inline and structure responses in a consulting-appropriate format (MECE where applicable). Low-confidence detection checks for hedging phrases. The system prompt explicitly requires citation of sources and strategic framing.

In [ ]:
# Milestone 3 — SOLUTION: Generation Layer with Citations

class RAGGenerator:
    def __init__(self, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        self.llm = ChatOpenAI(model=model, temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.system_prompt = """You are the McKinsey Knowledge Assistant. Answer consulting questions based ONLY on the provided knowledge base context.
Rules:
1. Cite sources using [Source: document_name] format
2. Structure answers in a clear, MECE (Mutually Exclusive, Collectively Exhaustive) format when applicable
3. If the context doesn't contain enough information, say "The knowledge base does not contain sufficient information to fully address this question."
4. Frame responses for an executive audience -- be concise, strategic, and actionable
5. Do not fabricate frameworks, data points, or recommendations beyond what the context provides
6. Where relevant, highlight key metrics, timelines, and success factors"""

    def generate(self, question, retrieved_chunks) -> Dict:
        # Format context with source tags
        context_parts = []
        sources_used = set()
        for chunk in retrieved_chunks:
            source = chunk['metadata'].get('source', 'unknown')
            practice = chunk['metadata'].get('practice_area', 'General')
            sources_used.add(source)
            context_parts.append(f"[Source: {source} | Practice: {practice}] {chunk['text']}")
        context = "\n\n".join(context_parts)
        
        # Generate
        start = time.time()
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=f"Knowledge Base Context:\n{context}\n\nConsulting Question: {question}")
        ])
        latency = (time.time() - start) * 1000
        
        content = response.content
        
        # Confidence detection
        low_confidence_phrases = ["does not contain sufficient", "insufficient information", "not enough context", "cannot determine", "no relevant information"]
        confidence = "low" if any(p in content.lower() for p in low_confidence_phrases) else "high"
        
        return {
            "content": content,
            "sources": list(sources_used),
            "confidence": confidence,
            "latency_ms": round(latency, 2),
            "context_chunks": len(retrieved_chunks)
        }


# Test with a consulting question
generator = RAGGenerator()
answer = generator.generate("What is McKinsey's approach to post-merger integration and what results can clients expect?", results)
print(f"Answer:\n{answer['content']}")
print(f"\nSources: {answer['sources']}")
print(f"Confidence: {answer['confidence']}")
print(f"Latency: {answer['latency_ms']}ms")

---
## Milestone 4: Production Wrapper -- Caching, Monitoring & Evaluation

Wrap everything in a production McKinsey Knowledge Assistant with caching, logging, cost tracking, and quality evaluation tailored to consulting standards.

Evaluation metrics go beyond generic RAG quality -- we measure **strategic relevance** (does the answer address the consulting question?), **faithfulness** (is it grounded in the knowledge base?), and **executive readiness** (is it structured, actionable, and suitable for partner/client review?).

> **Approach:** We compose all three milestone components into a single service class. The pipeline is: check cache -> retrieve -> generate -> evaluate -> cache & log. The dashboard aggregates cache hit rate, average latency, average consulting quality score, and usage patterns.

In [ ]:
# Milestone 4 — SOLUTION: Production McKinsey Knowledge Assistant

class ProductionRAGService:
    def __init__(self, documents):
        # Initialize components
        self.ingester = DocumentIngester(collection_name="mckinsey_prod")
        self.ingester.ingest(documents)
        self.retriever = RetrievalEngine(self.ingester.collection)
        self.generator = RAGGenerator()
        self.eval_llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        
        # Cache and monitoring
        self.cache = {}  # hash -> response
        self.logs = []
        self.quality_scores = []
    
    def _hash(self, text):
        return hashlib.md5(text.strip().lower().encode()).hexdigest()
    
    def _evaluate(self, question, answer, context_chunks):
        context = " ".join([c['text'] for c in context_chunks])
        metrics = {}
        for metric, prompt in {
            "strategic_relevance": f"Rate 1-5: How strategically relevant is this answer for a McKinsey consultant or partner asking the question? Does it address the core consulting need?\nQuestion: {question}\nAnswer: {answer}",
            "faithfulness": f"Rate 1-5: Is the answer fully supported by the knowledge base context? Are all claims grounded?\nContext: {context[:500]}\nAnswer: {answer}",
            "executive_readiness": f"Rate 1-5: Is this answer structured, concise, and actionable enough for executive or partner review? Does it follow consulting communication standards (e.g., MECE, SCR)?\nQuestion: {question}\nAnswer: {answer}"
        }.items():
            resp = self.eval_llm.invoke([
                SystemMessage(content="You are evaluating a McKinsey Knowledge Assistant response. Return ONLY a number 1-5."),
                HumanMessage(content=prompt)
            ])
            try:
                metrics[metric] = int(resp.content.strip())
            except:
                metrics[metric] = 3
        metrics["overall"] = round(np.mean(list(metrics.values())), 2)
        return metrics
    
    def query(self, question) -> Dict:
        start = time.time()
        
        # Check cache
        key = self._hash(question)
        if key in self.cache:
            cached = self.cache[key]
            latency = (time.time() - start) * 1000
            self.logs.append({"question": question, "cached": True, "latency_ms": latency})
            return {**cached, "cached": True, "latency_ms": round(latency, 2)}
        
        # Retrieve from knowledge base
        chunks = self.retriever.retrieve(question, k=3)
        
        # Generate executive-ready answer
        answer = self.generator.generate(question, chunks)
        
        # Evaluate consulting quality
        metrics = self._evaluate(question, answer["content"], chunks)
        self.quality_scores.append(metrics["overall"])
        
        latency = (time.time() - start) * 1000
        result = {
            "answer": answer["content"],
            "sources": answer["sources"],
            "confidence": answer["confidence"],
            "metrics": metrics,
            "cached": False,
            "latency_ms": round(latency, 2)
        }
        
        # Cache and log
        self.cache[key] = result
        self.logs.append({"question": question, "cached": False,
                          "latency_ms": latency, "quality": metrics["overall"]})
        
        return result
    
    def dashboard(self) -> Dict:
        total = len(self.logs)
        cached = sum(1 for l in self.logs if l.get("cached"))
        return {
            "total_queries": total,
            "cache_hit_rate": f"{cached/total*100:.1f}%" if total else "N/A",
            "avg_latency_ms": round(np.mean([l["latency_ms"] for l in self.logs]), 2) if self.logs else 0,
            "avg_consulting_quality": round(np.mean(self.quality_scores), 2) if self.quality_scores else 0,
            "cache_size": len(self.cache)
        }


# Test the McKinsey Knowledge Assistant
service = ProductionRAGService(test_docs)

# Realistic consulting questions a partner or EM might ask
questions = [
    "What is McKinsey's approach to post-merger integration?",
    "Summarize best practices for supply chain digitization in manufacturing",
    "What is McKinsey's approach to post-merger integration?",  # cached
    "How should I structure a CEO briefing for an operations transformation?"
]

for q in questions:
    result = service.query(q)
    cached_str = "CACHED" if result["cached"] else f"quality={result['metrics']['overall']}"
    print(f"[{cached_str:12s}] {q[:55]:55s} | {result['latency_ms']:.0f}ms")
    print(f"  Answer: {result['answer'][:120]}...\n")

print("--- McKinsey Knowledge Assistant Dashboard ---")
for k, v in service.dashboard().items():
    print(f"  {k}: {v}")

---
### Milestone 5: Multi-Turn Conversational Knowledge Assistant

Extend the Production RAG Service with conversational memory so consultants can ask follow-up questions. The system must rewrite follow-ups into self-contained queries before retrieval.

In [ ]:
# Milestone 5 — SOLUTION: Multi-Turn Conversational Knowledge Assistant

class ConversationalKnowledgeAssistant:
    """Extends ProductionRAGService with multi-turn conversation support."""
    
    def __init__(self, rag_service):
        self.rag = rag_service
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.sessions = {}  # session_id -> conversation history
    
    def _rewrite_query(self, question, history):
        """Rewrite a follow-up question to be self-contained using conversation history."""
        if not history:
            return question
        
        history_text = "\n".join([f"{h['role'].upper()}: {h['content'][:200]}" for h in history[-6:]])
        response = self.llm.invoke([
            SystemMessage(content=(
                "You are a query rewriter for a McKinsey Knowledge Assistant. "
                "Given a conversation history and a follow-up question, rewrite the follow-up "
                "to be a self-contained search query. Include all necessary consulting context. "
                "Return ONLY the rewritten query."
            )),
            HumanMessage(content=f"Conversation:\n{history_text}\n\nFollow-up: {question}\n\nRewritten query:")
        ])
        return response.content.strip()
    
    def ask(self, question, session_id="default"):
        """Ask a question with multi-turn conversation support."""
        if session_id not in self.sessions:
            self.sessions[session_id] = []
        
        history = self.sessions[session_id]
        search_query = self._rewrite_query(question, history)
        
        # Use the production RAG service for retrieval and generation
        result = self.rag.query(search_query)
        
        # Update conversation history
        history.append({"role": "user", "content": question})
        history.append({"role": "assistant", "content": result["answer"]})
        
        return {
            "turn": len(history) // 2,
            "original_question": question,
            "search_query": search_query,
            **result
        }
    
    def get_session_summary(self, session_id="default"):
        """Return a summary of the conversation session."""
        history = self.sessions.get(session_id, [])
        return {
            "session_id": session_id,
            "total_turns": len(history) // 2,
            "questions": [h["content"] for h in history if h["role"] == "user"]
        }


# Test with the production service from Milestone 4
# assistant = ConversationalKnowledgeAssistant(service)
# r1 = assistant.ask("What is McKinsey's approach to post-merger integration?")
# print(f"Turn {r1['turn']}: {r1['answer'][:150]}...")
# r2 = assistant.ask("What are the biggest risks?")  # Follow-up
# print(f"Turn {r2['turn']} (rewritten: {r2['search_query']}): {r2['answer'][:150]}...")
# r3 = assistant.ask("How do we mitigate those?")  # Another follow-up
# print(f"Turn {r3['turn']} (rewritten: {r3['search_query']}): {r3['answer'][:150]}...")
# print(f"\nSession summary: {assistant.get_session_summary()}")


---
### Milestone 6: A/B Testing and Configuration Comparison

Build an evaluation harness that compares different RAG configurations (chunk sizes, retrieval methods, prompts) on the same question set to identify the best setup for the McKinsey Knowledge Assistant.

In [ ]:
# Milestone 6 — SOLUTION: A/B Testing and Configuration Comparison

class RAGConfigTester:
    """Compares different RAG configurations on the same question set."""
    
    def __init__(self):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.results = {}  # config_name -> list of evaluation results
    
    def _build_rag(self, documents, chunk_size, chunk_overlap, k):
        """Build a simple RAG pipeline with specified configuration."""
        embed_client = openai.OpenAI()
        chroma = chromadb.Client()
        
        # Chunk documents
        splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
        chunks = []
        for doc in documents:
            chunks.extend(splitter.split_text(doc))
        
        # Index
        coll_name = f"ab_test_{chunk_size}_{chunk_overlap}_{hash(str(chunk_size)) % 10000}"
        coll = chroma.create_collection(name=coll_name, metadata={"hnsw:space": "cosine"})
        embeddings = [e.embedding for e in embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=chunks
        ).data]
        coll.add(documents=chunks, embeddings=embeddings, ids=[f"c_{i}" for i in range(len(chunks))])
        
        return coll, embed_client, chunks
    
    def _query(self, coll, embed_client, question, k=3):
        """Run a RAG query."""
        q_emb = embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[question]
        ).data[0].embedding
        results = coll.query(query_embeddings=[q_emb], n_results=k)
        context = "\n\n".join(results["documents"][0])
        response = self.llm.invoke([
            SystemMessage(content="You are a McKinsey knowledge assistant. Answer concisely using the context."),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
        ])
        return response.content, context
    
    def _evaluate(self, question, answer, context):
        """Evaluate answer quality using LLM-as-judge."""
        response = self.llm.invoke([
            SystemMessage(content=(
                "Rate the answer quality on 3 dimensions (1-5 each). "
                "Return ONLY JSON: {\"relevance\": N, \"faithfulness\": N, \"completeness\": N}"
            )),
            HumanMessage(content=f"Question: {question}\nContext: {context[:500]}\nAnswer: {answer}")
        ])
        try:
            return json.loads(response.content)
        except:
            return {"relevance": 3, "faithfulness": 3, "completeness": 3}
    
    def test_config(self, config_name, documents, questions, chunk_size=200, chunk_overlap=30, k=3):
        """Test a specific configuration on all questions."""
        coll, embed_client, chunks = self._build_rag(documents, chunk_size, chunk_overlap, k)
        print(f"\nConfig \"{config_name}\": {len(chunks)} chunks (size={chunk_size}, overlap={chunk_overlap}, k={k})")
        
        evals = []
        for q in questions:
            answer, context = self._query(coll, embed_client, q, k)
            metrics = self._evaluate(q, answer, context)
            metrics["question"] = q
            evals.append(metrics)
        
        self.results[config_name] = evals
        return evals
    
    def compare(self):
        """Compare all tested configurations and recommend the best one."""
        summary = {}
        for name, evals in self.results.items():
            avg_scores = {}
            for metric in ["relevance", "faithfulness", "completeness"]:
                avg_scores[metric] = round(np.mean([e[metric] for e in evals]), 2)
            avg_scores["overall"] = round(np.mean(list(avg_scores.values())), 2)
            summary[name] = avg_scores
        
        # Find best config
        best = max(summary.items(), key=lambda x: x[1]["overall"])
        
        print("\n=== A/B Test Results ===")
        for name, scores in summary.items():
            marker = " *** BEST" if name == best[0] else ""
            print(f"  {name}: {scores}{marker}")
        
        return {"summary": summary, "recommended": best[0]}


# Test
# tester = RAGConfigTester()
# test_docs = [...]
# test_questions = [
#     "What are best practices for post-merger integration?",
#     "How does digital transformation improve supply chain efficiency?",
# ]
# tester.test_config("small_chunks", test_docs, test_questions, chunk_size=100, chunk_overlap=20)
# tester.test_config("large_chunks", test_docs, test_questions, chunk_size=300, chunk_overlap=50)
# result = tester.compare()
# print(f"\nRecommended config: {result['recommended']}")


---
## Capstone Summary

Students built a **production-grade McKinsey Knowledge Assistant** with:

1. **Ingestion** -- Smart chunking of consulting assets (frameworks, playbooks, industry reports) with rich metadata (practice area, industry, engagement type, confidentiality level)
2. **Retrieval** -- Multi-query expansion and LLM reranking tuned for consulting questions from partners and engagement managers
3. **Generation** -- Executive-ready answers with source citations, MECE structure, and confidence flagging
4. **Production** -- Caching, monitoring, and automated evaluation measuring strategic relevance, faithfulness, and executive readiness

**Up Next:** Session 4 -- Cross-track presentations, governance, and closing.